# 03. Query 특성별 강점 분석

**팀원 A 핵심 기여** — 단순 순위 비교가 아니라 '어떤 질문에서 어떤 모델이 강한가' 분석

분석 기준 (data_handoff 문서 기준):
- `lexical` : 정확한 키워드 매칭이 중요한 질문 → BM25 강점 예상
- `semantic` : 의미 이해가 필요한 질문 → Embedding 강점 예상
- `has_korean_entity` : 삼성·SKT·KT 등 한국어 고유명사 포함 → ko-sroberta 강점 예상
- `has_english_term` : OpenAI·NVIDIA·GPT 등 영문 기술용어 포함 → OpenAI·BGE-M3 강점 예상
- 카테고리별 (LLM / INFRA / AI Business / Telco)

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# ── QA 27개 특성 태그 (수동 레이블링) ──
# data_handoff + qa_benchmark_seed 두 문서 참고해서 태그 부여
#
# query_style:
#   lexical  = 특정 고유명사/키워드가 정답 문서에 그대로 있어야 하는 질문
#   semantic = 개념 이해·트렌드 종합이 필요한 질문
#
# has_english_term: OpenAI, NVIDIA, GPT, LLM, HBM 등 영문 기술용어 포함
# has_korean_entity: 삼성전자, SKT, KT, LG, 한국 정부 등 한국어 고유명사 포함

QA_CHARACTERISTICS = {
    # LLM 카테고리
    'L1': {'style': 'lexical',  'has_english': True,  'has_korean_entity': False,
           'note': 'OpenAI 신모델 — 고유명사 키워드 매칭'},
    'L2': {'style': 'semantic', 'has_english': True,  'has_korean_entity': False,
           'note': 'AI 에이전트 트렌드 — 종합 이해 필요'},
    'L3': {'style': 'lexical',  'has_english': True,  'has_korean_entity': False,
           'note': 'Anthropic 매출 — 회사명 키워드'},
    'L4': {'style': 'semantic', 'has_english': True,  'has_korean_entity': False,
           'note': 'Gemini 전략 — 의미 이해 + 영문'},
    'L5': {'style': 'semantic', 'has_english': True,  'has_korean_entity': False,
           'note': '멀티모달 트렌드 — 다수 기사 종합'},
    'L6': {'style': 'semantic', 'has_english': True,  'has_korean_entity': False,
           'note': '오픈소스 LLM 동향 — 영문 혼합 트렌드'},
    # INFRA 카테고리
    'I1': {'style': 'lexical',  'has_english': False, 'has_korean_entity': True,
           'note': '삼성전자 — 한국어 회사명 키워드'},
    'I2': {'style': 'lexical',  'has_english': True,  'has_korean_entity': True,
           'note': 'SK하이닉스·HBM — 한영 혼합 고유명사'},
    'I3': {'style': 'semantic', 'has_english': True,  'has_korean_entity': False,
           'note': 'NVIDIA 칩 동향 — 영문 기술용어 트렌드'},
    'I4': {'style': 'semantic', 'has_english': False, 'has_korean_entity': True,
           'note': '국내 반도체 스타트업 — 한국어 의미 종합'},
    'I5': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': 'AI 반도체 미중 경쟁 — 지정학적 트렌드'},
    'I6': {'style': 'semantic', 'has_english': True,  'has_korean_entity': False,
           'note': 'Edge AI·on-device — 영문 기술 트렌드'},
    # AI Business 카테고리
    'B1': {'style': 'lexical',  'has_english': True,  'has_korean_entity': False,
           'note': 'OpenAI IPO — 회사명 + 이벤트 키워드'},
    'B2': {'style': 'lexical',  'has_english': False, 'has_korean_entity': False,
           'note': 'AI M&A — 이벤트 키워드'},
    'B3': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': 'AI 스타트업 투자 트렌드 — 의미 종합'},
    'B4': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': '글로벌 AI 규제 — 정책 트렌드'},
    'B5': {'style': 'lexical',  'has_english': False, 'has_korean_entity': True,
           'note': '한국 정부 AI 정책 — 한국어 고유명사'},
    'B6': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': '생성형 AI 시장 — 트렌드 종합'},
    'B7': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': '글로벌 AI 허브 경쟁 — 의미 비교'},
    'B8': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': '빅테크 AI 투자 비교 — 다수 기업 비교'},
    'B9': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': 'AI 일자리 영향 — 사회적 트렌드'},
    # Telco 카테고리
    'T1': {'style': 'lexical',  'has_english': False, 'has_korean_entity': True,
           'note': 'SKT — 한국어 회사명 키워드'},
    'T2': {'style': 'lexical',  'has_english': False, 'has_korean_entity': True,
           'note': 'KT — 한국어 회사명 키워드'},
    'T3': {'style': 'lexical',  'has_english': False, 'has_korean_entity': True,
           'note': 'LG U+ — 한국어 회사명 키워드'},
    'T4': {'style': 'semantic', 'has_english': False, 'has_korean_entity': False,
           'note': '5G·6G AI 결합 — 기술 트렌드'},
    'T5': {'style': 'semantic', 'has_english': False, 'has_korean_entity': True,
           'note': '통신사 데이터센터 — 한국 기업 트렌드'},
    'T6': {'style': 'semantic', 'has_english': False, 'has_korean_entity': True,
           'note': '통신사 AI 에이전트 — 한국 기업 비교'},
}

print(f'태그 부여된 QA: {len(QA_CHARACTERISTICS)}개')
lexical_count  = sum(1 for v in QA_CHARACTERISTICS.values() if v['style'] == 'lexical')
semantic_count = sum(1 for v in QA_CHARACTERISTICS.values() if v['style'] == 'semantic')
english_count  = sum(1 for v in QA_CHARACTERISTICS.values() if v['has_english'])
korean_count   = sum(1 for v in QA_CHARACTERISTICS.values() if v['has_korean_entity'])
print(f'  lexical:           {lexical_count}개')
print(f'  semantic:          {semantic_count}개')
print(f'  영문 기술용어 포함: {english_count}개')
print(f'  한국어 고유명사:    {korean_count}개')

## 02_retrieval_baseline 결과 불러오기

02 노트북을 먼저 실행한 후 이 노트북을 실행하세요.

In [ ]:
import pandas as pd
import numpy as np
import json
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

corpus   = pd.read_parquet('../data/insk_corpus.parquet')
analyses = pd.read_parquet('../data/article_analyses.parquet')
emb_df   = pd.read_parquet('../data/article_embeddings.parquet')
qa_list  = [json.loads(l) for l in open('../data/human_qa_benchmark_v1.jsonl', encoding='utf-8')]

df = corpus.merge(analyses, on='article_id', how='left').reset_index(drop=True)
article_ids = df['article_id'].tolist()

def build_doc_text(row):
    tags = ' '.join(json.loads(row['tags'])) if pd.notna(row['tags']) else ''
    return f"{row['title']} {row['summary'] or ''} {row['insight'] or ''} {tags}"

doc_texts = df.apply(build_doc_text, axis=1).tolist()

# BM25
bm25 = BM25Okapi([t.split() for t in doc_texts])
def bm25_retrieve(q, k=10):
    scores = bm25.get_scores(q.split())
    top = np.argsort(scores)[::-1][:k]
    return [article_ids[i] for i in top]

# OpenAI Embedding (벡터 로드)
emb_df['embedding'] = emb_df['embedding_json'].apply(lambda s: np.array(json.loads(s), dtype=np.float32))
emb_merged = df[['article_id']].merge(emb_df[['article_id','embedding']], on='article_id', how='left')
openai_vectors = np.vstack(emb_merged['embedding'].values)
faiss.normalize_L2(openai_vectors)
index_openai = faiss.IndexFlatIP(1536)
index_openai.add(openai_vectors)

from openai import OpenAI
client = OpenAI()

def openai_retrieve(q, k=10):
    resp = client.embeddings.create(model='text-embedding-3-small', input=q)
    q_vec = np.array(resp.data[0].embedding, dtype=np.float32).reshape(1, -1)
    faiss.normalize_L2(q_vec)
    _, indices = index_openai.search(q_vec, k)
    return [article_ids[i] for i in indices[0]]

# Hybrid RRF
def rrf(rankings_list, rrf_k=60):
    scores = {}
    for rankings in rankings_list:
        for rank, aid in enumerate(rankings):
            scores[aid] = scores.get(aid, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.keys(), key=lambda x: -scores[x])

def hybrid_retrieve(q, k=10):
    return rrf([bm25_retrieve(q, k), openai_retrieve(q, k)])[:k]

print('모델 로드 완료')

In [ ]:
# BGE-M3, ko-sroberta (02에서 이어서 사용할 경우 주석 해제)
print('BGE-M3 로드 중...')
model_bge = SentenceTransformer('BAAI/bge-m3')
bge_vectors = model_bge.encode(doc_texts, batch_size=32, show_progress_bar=True,
                               normalize_embeddings=True).astype(np.float32)
index_bge = faiss.IndexFlatIP(bge_vectors.shape[1])
index_bge.add(bge_vectors)

print('ko-sroberta 로드 중...')
model_ko = SentenceTransformer('jhgan/ko-sroberta-multitask')
ko_vectors = model_ko.encode(doc_texts, batch_size=32, show_progress_bar=True,
                              normalize_embeddings=True).astype(np.float32)
index_ko = faiss.IndexFlatIP(ko_vectors.shape[1])
index_ko.add(ko_vectors)

def bge_retrieve(q, k=10):
    q_vec = model_bge.encode([q], normalize_embeddings=True).astype(np.float32)
    _, idx = index_bge.search(q_vec, k)
    return [article_ids[i] for i in idx[0]]

def ko_retrieve(q, k=10):
    q_vec = model_ko.encode([q], normalize_embeddings=True).astype(np.float32)
    _, idx = index_ko.search(q_vec, k)
    return [article_ids[i] for i in idx[0]]

print('전체 모델 준비 완료')

## 핵심 분석 — Query 특성별 Recall@5

In [ ]:
def recall_at_k(retrieved, gold, k=5):
    if not gold: return None
    return len(set(retrieved[:k]) & set(gold)) / len(gold)

def mrr(retrieved, gold):
    if not gold: return None
    for rank, rid in enumerate(retrieved, 1):
        if rid in gold:
            return 1.0 / rank
    return 0.0

MODELS = {
    'BM25':        bm25_retrieve,
    'OpenAI Emb':  openai_retrieve,
    'BGE-M3':      bge_retrieve,
    'ko-sroberta': ko_retrieve,
    'Hybrid RRF':  hybrid_retrieve,
}

# 평가 대상: Strict + Trend (Negative는 정답 없음)
eval_qa = [q for q in qa_list if q['type'] != 'Negative']

# 각 QA에 특성 태그 붙이기
rows = []
for q in eval_qa:
    char = QA_CHARACTERISTICS.get(q['id'], {})
    row = {
        'id':          q['id'],
        'question':    q['question'],
        'category':    q['category'],
        'type':        q['type'],
        'gold':        q['gold_articles'],
        'style':       char.get('style', 'unknown'),
        'has_english': char.get('has_english', False),
        'has_korean':  char.get('has_korean_entity', False),
    }
    # 각 모델 Recall@5 계산
    for model_name, fn in MODELS.items():
        retrieved = fn(q['question'])
        row[f'recall_{model_name}'] = recall_at_k(retrieved, q['gold_articles'])
        row[f'mrr_{model_name}']    = mrr(retrieved, q['gold_articles'])
    rows.append(row)

result_df = pd.DataFrame(rows)
print(f'평가 완료: {len(result_df)}개 QA')
result_df[['id','category','type','style','has_english','has_korean']].head(10)

### 분석 1. Lexical vs Semantic — 어떤 모델이 각 유형에 강한가

In [ ]:
print('='*60)
print('Lexical 질문 vs Semantic 질문 — Recall@5 평균')
print('='*60)
print(f'{"모델":<14} {"Lexical":>10} {"Semantic":>10} {"Lexical 강점"}') 
print('-'*60)

for model_name in MODELS:
    col = f'recall_{model_name}'
    lex  = result_df[result_df['style']=='lexical' ][col].dropna().mean()
    sem  = result_df[result_df['style']=='semantic'][col].dropna().mean()
    diff = lex - sem
    label = '← BM25 기대 영역' if model_name == 'BM25' and diff > 0 else \
            '← Emb 기대 영역' if 'Emb' in model_name and diff < 0 else ''
    print(f'{model_name:<14} {lex:>10.3f} {sem:>10.3f}  Δ={diff:+.3f} {label}')

print()
print(f'Lexical  QA 수: {len(result_df[result_df["style"]=="lexical"])}개')
print(f'Semantic QA 수: {len(result_df[result_df["style"]=="semantic"])}개')

In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(MODELS))
w = 0.35
model_names = list(MODELS.keys())
colors = ['#4C72B0', '#DD8452']

lex_vals = [result_df[result_df['style']=='lexical' ][f'recall_{m}'].dropna().mean() for m in model_names]
sem_vals = [result_df[result_df['style']=='semantic'][f'recall_{m}'].dropna().mean() for m in model_names]

b1 = ax.bar(x - w/2, lex_vals, w, label='Lexical (키워드 중심)', color=colors[0])
b2 = ax.bar(x + w/2, sem_vals, w, label='Semantic (의미 이해 필요)', color=colors[1])

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Recall@5')
ax.set_title('Query 특성별 성능: Lexical vs Semantic', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

### 분석 2. 한국어 고유명사 vs 영문 기술용어

In [ ]:
print('='*60)
print('한국어 고유명사 포함 vs 영문 기술용어 포함 — Recall@5')
print('='*60)
print(f'{"모델":<14} {"한국어 엔티티":>13} {"영문 기술용어":>13}')
print('-'*60)

for model_name in MODELS:
    col = f'recall_{model_name}'
    kor = result_df[result_df['has_korean']==True ][col].dropna().mean()
    eng = result_df[result_df['has_english']==True][col].dropna().mean()
    print(f'{model_name:<14} {kor:>13.3f} {eng:>13.3f}')

print()
print(f'한국어 고유명사 포함 QA: {result_df["has_korean"].sum()}개  (SKT·KT·삼성 등)')
print(f'영문 기술용어 포함 QA:   {result_df["has_english"].sum()}개  (OpenAI·NVIDIA·GPT 등)')

### 분석 3. 카테고리별 (LLM / INFRA / AI Business / Telco)

In [ ]:
print('='*60)
print('카테고리별 Recall@5 — 어느 도메인에서 검색이 잘 되는가')
print('='*60)

categories = result_df['category'].unique()
cat_data = {}

for cat in ['LLM', 'INFRA', 'AI Business', 'Telco']:
    cat_df = result_df[result_df['category']==cat]
    if len(cat_df) == 0: continue
    print(f'\n  [{cat}] — {len(cat_df)}개 QA')
    for model_name in MODELS:
        col = f'recall_{model_name}'
        val = cat_df[col].dropna().mean()
        bar = '█' * int(val * 20)
        print(f'    {model_name:<14} {val:.3f} {bar}')
    cat_data[cat] = {m: result_df[result_df['category']==cat][f'recall_{m}'].dropna().mean()
                     for m in MODELS}

In [ ]:
# 카테고리별 히트맵
heat_data = pd.DataFrame(cat_data).T  # 행=카테고리, 열=모델

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heat_data.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels(list(MODELS.keys()), rotation=30, ha='right')
ax.set_yticks(range(len(heat_data)))
ax.set_yticklabels(heat_data.index)

for i in range(len(heat_data)):
    for j in range(len(MODELS)):
        val = heat_data.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color='white' if val > 0.5 else 'black', fontsize=11, fontweight='bold')

plt.colorbar(im, ax=ax, label='Recall@5')
ax.set_title('카테고리 × 모델 성능 히트맵 (Recall@5)', fontsize=13)
plt.tight_layout()
plt.show()

### 분석 4. QA 유형별 (Strict vs Trend)

In [ ]:
print('='*60)
print('QA 유형별 성능: Strict(정답 명확) vs Trend(종합)')
print('='*60)
print(f'{"모델":<14} {"Strict":>9} {"Trend":>9} {"Strict 우위"}')
print('-'*60)

for model_name in MODELS:
    col = f'recall_{model_name}'
    s = result_df[result_df['type']=='Strict'][col].dropna().mean()
    t = result_df[result_df['type']=='Trend' ][col].dropna().mean()
    diff = s - t
    print(f'{model_name:<14} {s:>9.3f} {t:>9.3f}  Δ={diff:+.3f}')

print()
print('해석 가이드:')
print('  Strict에서 높을수록 → 정확 키워드 매칭 강점 (BM25 기대)')
print('  Trend에서 높을수록  → 개념·맥락 이해 강점 (Embedding 기대)')

### 분석 5. 질문별 상세 — 어떤 질문에서 어떤 모델이 이겼나

In [ ]:
# 각 QA마다 '최고 모델' 찾기
recall_cols = [f'recall_{m}' for m in MODELS]
result_df['best_model'] = result_df[recall_cols].idxmax(axis=1).str.replace('recall_', '')
result_df['best_score'] = result_df[recall_cols].max(axis=1)

print('=== 질문별 최고 성능 모델 ===')
print(f'{"ID":<4} {"카테고리":<12} {"스타일":<9} {"최고모델":<14} {"Recall@5":<10} 질문')
print('-'*80)
for _, row in result_df.sort_values('category').iterrows():
    print(f'{row["id"]:<4} {row["category"]:<12} {row["style"]:<9} {row["best_model"]:<14} {row["best_score"]:<10.3f} {row["question"][:30]}')

print()
print('=== 모델별 1등 횟수 ===')
print(result_df['best_model'].value_counts())

## 최종 발표용 인사이트 정리

In [ ]:
print('='*65)
print('팀원 A 핵심 발표 인사이트')
print('='*65)
print()

# 각 특성별 최고 모델 자동 추출
for label, mask in [
    ('Lexical 질문',           result_df['style']=='lexical'),
    ('Semantic 질문',          result_df['style']=='semantic'),
    ('한국어 고유명사 포함',    result_df['has_korean']==True),
    ('영문 기술용어 포함',      result_df['has_english']==True),
]:
    sub = result_df[mask]
    best = {m: sub[f'recall_{m}'].dropna().mean() for m in MODELS}
    top_model = max(best, key=best.get)
    top_val   = best[top_model]
    print(f'  {label:<20}: {top_model} 최고  (Recall@5={top_val:.3f})')

print()
print('이 분석이 발표의 핵심입니다.')
print('"누가 1등이 아니라, 어떤 상황에서 어떤 모델을 쓰면 되는지"를 보여주는 것')